In [196]:
import pickle
import json
import numpy as np
from matplotlib import pyplot as plt
from scipy.ndimage import uniform_filter
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

In [197]:
def snr(clean, noisy):
    noise = clean.astype(float) - noisy.astype(float)
    power_signal = np.sum(clean.astype(float)**2)
    power_noise  = np.sum(noise**2)
    return 10 * np.log10(power_signal / power_noise)

def lee_filter(img, size=7, noise_var=None, printTrue=False):
    """
    img : float32 entre 0 et 1
    size : taille du voisinage (3,5,7…)
    """
    # moyenne locale
    mean = uniform_filter(img, size)
    # variance locale
    mean_sq = uniform_filter(img**2, size)
    var = mean_sq - mean**2

    # variance du bruit (supposée)
    noise_var = np.mean(var) if noise_var is None else noise_var
    if printTrue:
        print("Estimated noise variance =", noise_var)
    # poids
    w = var / (var + noise_var)
    
    # estimation Lee
    img_filtered = mean + w * (img - mean)

    return img_filtered

def estimate_speckle_var_from_patches(
    images,
    patch_size=16,
    target_mean=0.5,
    tol=0.05,
    max_patches_per_image=200,
    seed=0
):
    rng = np.random.default_rng(seed)
    speckle_vars = []

    for img in images:
        H, W = img.shape
        if H < patch_size or W < patch_size:
            continue

        for _ in range(max_patches_per_image):
            i = rng.integers(0, H - patch_size + 1)
            j = rng.integers(0, W - patch_size + 1)

            patch = img[i:i+patch_size, j:j+patch_size]
            m = patch.mean()

            # condition mean ≈ 0.5
            if abs(m - target_mean) <= tol:
                vY = patch.var()
                # Var(N) ≈ Var(Y) / m^2
                vN = vY / (m**2 + 1e-12)
                speckle_vars.append(vN)

    if len(speckle_vars) == 0:
        raise RuntimeError("Aucun patch ne respecte la condition sur le mean.")

    return float(np.mean(speckle_vars)), len(speckle_vars)

def estimate_noise_var(images, size=7):
    noise_vars = []

    for img in images:
        # moyenne locale
        mean = uniform_filter(img, size)
        mean_sq = uniform_filter(img**2, size)
        var = mean_sq - mean**2     # variance locale image
        noise_vars.append(np.mean(var))  # variance moyenne de l'image

    # variance de bruit globale sur tout le dataset
    return float(np.mean(noise_vars))
    

In [198]:

with open('../../../ETL/data/pickles/processed.pkl', 'rb') as file:
	data = pickle.load(file)



with open('results_leeFilter.json', 'r') as file :
    results = json.load(file)

In [199]:
with open('../../../ETL/data/L.txt', 'r') as file:
    l = file.readline()

L = int(l)

In [200]:
L

100

In [212]:
# Estimer noise
images_noisy = data['bsd68']['noisy']  # liste d’images float32 [0,1]

vN_est, n_patches = estimate_speckle_var_from_patches(
    images_noisy,
    patch_size=32,
    target_mean=0.5,
    tol=0.05,
    max_patches_per_image=200,
    seed=1520
)

print("Méthode Amélioré : Var(N) estimée =", vN_est, "à partir de", n_patches, "patches")




# images_100 = data['div2k']['lr4']['DIV2K_train_LR_bicubic']['X4']['noisy']
images_100 = data['bsd68']['noisy']
noise_var = estimate_noise_var(images_100, size=5)

print("Methode simple : Estimated noise variance =", noise_var)


Méthode Amélioré : Var(N) estimée = 0.10054097465817796 à partir de 2576 patches
Methode simple : Estimated noise variance = 0.008254453539848328


In [202]:
r = []
for i in range(12):
    r.append({})
    for size in [3, 5, 7, 9, 11] : 
        caseValue = 'size : ' + str(size)
        r[-1][caseValue] = {}
        img = data['set12']['clean'][i]
        noisy_img = data['set12']['noisy'][i]
        r[-1][caseValue]['snr'] = snr(img, noisy_img)

        lee_img = lee_filter(noisy_img, size=size)
        
        r[-1][caseValue]['psnr'] = psnr(img, lee_img, data_range=1.0)
        r[-1][caseValue]['ssim'] = ssim(img, lee_img, data_range=1.0)
        
        lee_img_vN = lee_filter(noisy_img, size=size, noise_var=vN_est)
        
        r[-1][caseValue]['psnr_vN'] = psnr(img, lee_img_vN, data_range=1.0)
        r[-1][caseValue]['ssim_vN'] = ssim(img, lee_img_vN, data_range=1.0)


In [203]:
rs = {i : 
    {
        'snr': np.mean([j[i]['snr'] for j in r]),
        'psnr': np.mean([j[i]['psnr'] for j in r]),
        'ssim': np.mean([j[i]['ssim'] for j in r]),
        'psnr_vN': np.mean([j[i]['psnr_vN'] for j in r]),
        'ssim_vN': np.mean([j[i]['ssim_vN'] for j in r]),
        'estimated Var vN' : vN_est,
        'estimated Var classic' : noise_var,
        'real var' : 1/L
    } for i in r[0].keys()} 

In [204]:
#rs

In [205]:
results.keys()

dict_keys(['1', '2', '3', '4', '5', '7', '10', '15', '20', '30', '50', '13', '16', '25', '40', '70'])

In [206]:
results[str(L)] = rs

In [207]:
results.keys()

dict_keys(['1', '2', '3', '4', '5', '7', '10', '15', '20', '30', '50', '13', '16', '25', '40', '70', '100'])

In [208]:
with open('results_leeFilter.json', 'w') as file:
    json.dump(results, file)